In [1]:
import pandas as pd

data = pd.read_csv("../../processed/cedar_final.csv")
print("total peptides:", len(data))
data.head()

total peptides: 6499


,epitope_url,mt_seq,wt_seq,mutation,epitope_relation,source_molecule,label,has_wildtype,mt_length,wt_length,epitope_id,hla_allele,n_alleles,assay_any_positive,n_assays_total
0,https://cedar.iedb.org/epitope/551,ACDPHSGHFV,ARDPHSGHFV,R24C,in-frame neo-epitope,NaN,1,True,10,10,551,HLA-A*02:01,2.0,True,10.0
1,https://cedar.iedb.org/epitope/3147,AMLGTHTMEV,AMLSPHAMDV,NaN,in-frame neo-epitope,NaN,1,True,10,10,3147,HLA-A*02:01,1.0,True,7.0
2,https://cedar.iedb.org/epitope/14672,EVDPIGHLY,EVVPISHLY,NaN,in-frame neo-epitope,NaN,1,True,9,9,14672,HLA-A*01:01;HLA-A*11:01;HLA-B*35:01;HLA-C*05:01,5.0,True,132.0
3,https://cedar.iedb.org/epitope/23214,GVYDGEEHSV,GAYDGEEH,NaN,in-frame neo-epitope,NaN,1,True,10,8,23214,HLA-A*02:01,1.0,True,11.0
4,https://cedar.iedb.org/epitope/28958,ITKKVADLVGF,KKVADLI,NaN,in-frame neo-epitope,NaN,1,True,11,7,28958,HLA-B*57:01;HLA-B*57:02,2.0,True,7.0


In [2]:
#keep only rows with 4 digit hla
predictable_rows = [] 

for i, row in data.iterrows(): 
    hla_str = row["hla_allele"]

    #skip missing
    if not isinstance(hla_str, str): 
        continue

    #split many hla
    alleles = hla_str.split(";")

    #check if at least one is 4 digit
    valid_hla = False
    for allele in alleles: 
        if "*" in allele and ":" in allele: 
            valid_hla = True
            break
    if valid_hla: 
        predictable_rows.append(row) 

#row back to df
predictable = pd.DataFrame(predictable_rows)
print("predictable peptides:", len(predictable))
print("peptides dropped:", len(data) - len(predictable))

predictable peptides: 6328
peptides dropped: 171


In [3]:
#exand to one row/hla

rows = [] 
for i, row in predictable.iterrows(): 
    alleles = row["hla_allele"].split(";")
    for allele in alleles: 
        #output row
        rows.append({
            "epitope_id": row["epitope_id"],
            "hla": allele, 
            "mt_seq": row["mt_seq"],
            "wt_seq": row["wt_seq"],
            "label": row["label"], 
            "mt_length": row["mt_length"],
        })

#long format table
predict_input = pd.DataFrame(rows)
print("total rows:", len(predict_input))
predict_input.head()

total rows: 6520


,epitope_id,hla,mt_seq,wt_seq,label,mt_length
0,551,HLA-A*02:01,ACDPHSGHFV,ARDPHSGHFV,1,10
1,3147,HLA-A*02:01,AMLGTHTMEV,AMLSPHAMDV,1,10
2,14672,HLA-A*01:01,EVDPIGHLY,EVVPISHLY,1,9
3,14672,HLA-A*11:01,EVDPIGHLY,EVVPISHLY,1,9
4,14672,HLA-B*35:01,EVDPIGHLY,EVVPISHLY,1,9


In [4]:
#mhc flurry predictions

from mhcflurry import Class1PresentationPredictor
predictor = Class1PresentationPredictor.load()

In [5]:
#allele dict
#mhcflurry format: {sample:[alleles]}
#each allele should be independent 
#so each unique allele is a sample 

unique_alleles = predict_input["hla"].unique()
alleles_dict = {}
for allele in unique_alleles: 
    alleles_dict[allele] = [allele] 

print(alleles_dict)

{'HLA-A*02:01': ['HLA-A*02:01'], 'HLA-A*01:01': ['HLA-A*01:01'], 'HLA-A*11:01': ['HLA-A*11:01'], 'HLA-B*35:01': ['HLA-B*35:01'], 'HLA-C*05:01': ['HLA-C*05:01'], 'HLA-B*57:01': ['HLA-B*57:01'], 'HLA-B*57:02': ['HLA-B*57:02'], 'HLA-A*24:02': ['HLA-A*24:02'], 'HLA-A*03:01': ['HLA-A*03:01'], 'HLA-B*44:03': ['HLA-B*44:03'], 'HLA-B*15:01': ['HLA-B*15:01'], 'HLA-B*07:02': ['HLA-B*07:02'], 'HLA-B*44:02': ['HLA-B*44:02'], 'HLA-B*38:01': ['HLA-B*38:01'], 'HLA-C*14:02': ['HLA-C*14:02'], 'HLA-B*27:05': ['HLA-B*27:05'], 'HLA-A*68:02': ['HLA-A*68:02'], 'HLA-B*52:01': ['HLA-B*52:01'], 'HLA-A*02:05': ['HLA-A*02:05'], 'HLA-C*06:02': ['HLA-C*06:02'], 'HLA-A*30:01': ['HLA-A*30:01'], 'HLA-B*08:01': ['HLA-B*08:01'], 'HLA-B*13:02': ['HLA-B*13:02'], 'HLA-B*51:01': ['HLA-B*51:01'], 'HLA-A*02:03': ['HLA-A*02:03'], 'HLA-C*03:03': ['HLA-C*03:03'], 'HLA-A*30:02': ['HLA-A*30:02'], 'HLA-A*25:01': ['HLA-A*25:01'], 'HLA-A*23:01': ['HLA-A*23:01'], 'HLA-A*02:11': ['HLA-A*02:11'], 'HLA-A*02:06': ['HLA-A*02:06'], 'HLA-C*

In [6]:
#each peptide has sample name matches allele
sample_names = predict_input["hla"].tolist()

mt_results = predictor.predict(
    peptides=predict_input["mt_seq"].tolist(), 
    alleles=alleles_dict, 
    sample_names = sample_names, 
    include_affinity_percentile=True, 
    verbose=1
)

print(list(mt_results.columns))
mt_results.head()

Predicting processing.


100%|██████████| 1/1 [00:01<00:00,  1.36s/it]


Predicting affinities.


100%|██████████| 97/97 [00:03<00:00, 29.61it/s]

['peptide', 'peptide_num', 'sample_name', 'affinity', 'best_allele', 'affinity_percentile', 'processing_score', 'presentation_score', 'presentation_percentile']


,peptide,peptide_num,sample_name,affinity,best_allele,affinity_percentile,processing_score,presentation_score,presentation_percentile
0,ACDPHSGHFV,0,HLA-A*02:01,3125.736683,HLA-A*02:01,2.796875,0.174882,0.053443,4.807092
1,AMLGTHTMEV,1,HLA-A*02:01,37.507257,HLA-A*02:01,0.260250,0.112602,0.768811,0.326821
2,EVDPIGHLY,2,HLA-A*01:01,33.501012,HLA-A*01:01,0.007125,0.891796,0.983180,0.004511
3,EVDPIGHLY,3,HLA-A*11:01,942.095939,HLA-A*11:01,1.615250,0.891796,0.695778,0.443832
4,EVDPIGHLY,4,HLA-B*35:01,64.056180,HLA-B*35:01,0.173750,0.891796,0.968890,0.021413


In [7]:
#sanity check
#do rows match? 
order_check = (mt_results["sample_name"].values == predict_input["hla"].values).all()
print("order check:", order_check)

#do peptides match?
pep_check = (mt_results["peptide"].values == predict_input["mt_seq"].values).all()
print("peptide check:", pep_check)

order check: True
peptide check: True


In [8]:
#summary stats
print(mt_results["affinity"].describe)
print(mt_results["affinity_percentile"].describe())
print(mt_results["presentation_score"].describe())

<bound method NDFrame.describe of 0        3125.736683
1          37.507257
2          33.501012
3         942.095939
4          64.056180
            ...     
6515    15036.288509
6516    16469.806418
6517    15264.953710
6518     6894.464435
6519      369.122514
Name: affinity, Length: 6520, dtype: float64>
count    6520.000000
mean        1.673119
std         4.470256
min         0.000000
25%         0.113250
50%         0.397062
75%         1.238906
max        80.005750
Name: affinity_percentile, dtype: float64
count    6520.000000
mean        0.551102
std         0.344781
min         0.003170
25%         0.198457
50%         0.640775
75%         0.870307
max         0.995949
Name: presentation_score, dtype: float64


In [9]:
#how many presernted at 2% threshold? 
n_presented = (mt_results["presentation_percentile"] <= 2).sum()
print("n presented at 2% threshold:", n_presented, "out of", len(mt_results))
print("percent:", n_presented / len(mt_results))

n presented at 2% threshold: 4973 out of 6520
percent: 0.7627300613496932


In [10]:
#check for NAns
print(mt_results.isna().sum())

peptide                    0
peptide_num                0
sample_name                0
affinity                   0
best_allele                0
affinity_percentile        0
processing_score           0
presentation_score         0
presentation_percentile    0
dtype: int64


In [11]:
##repeat for wt peptides

wt_results = predictor.predict(
    peptides = predict_input["wt_seq"].tolist(),
    alleles = alleles_dict,
    sample_names = sample_names, #same hla so same names
    include_affinity_percentile=True,
    verbose=1
)
print(wt_results.columns)
wt_results.head()

Predicting processing.


100%|██████████| 1/1 [00:00<00:00,  1.74it/s]


Predicting affinities.


100%|██████████| 97/97 [00:02<00:00, 46.97it/s]

Index(['peptide', 'peptide_num', 'sample_name', 'affinity', 'best_allele',
       'affinity_percentile', 'processing_score', 'presentation_score',
       'presentation_percentile'],
      dtype='object')


,peptide,peptide_num,sample_name,affinity,best_allele,affinity_percentile,processing_score,presentation_score,presentation_percentile
0,ARDPHSGHFV,0,HLA-A*02:01,13379.793366,HLA-A*02:01,7.050250,0.397545,0.029345,8.269212
1,AMLSPHAMDV,1,HLA-A*02:01,104.448134,HLA-A*02:01,0.625750,0.034954,0.483023,0.858478
2,EVVPISHLY,2,HLA-A*01:01,164.471650,HLA-A*01:01,0.231000,0.962785,0.941244,0.060136
3,EVVPISHLY,3,HLA-A*11:01,1086.646020,HLA-A*11:01,1.703250,0.962785,0.719056,0.406060
4,EVVPISHLY,4,HLA-B*35:01,50.299185,HLA-B*35:01,0.111375,0.962785,0.980633,0.007255


In [12]:
print("total predictions:", len(mt_results))
print(wt_results.isna().sum()) #check nans

total predictions: 6520
peptide                    0
peptide_num                0
sample_name                0
affinity                   0
best_allele                0
affinity_percentile        0
processing_score           0
presentation_score         0
presentation_percentile    0
dtype: int64


In [13]:
#how many presernted at 2% threshold? 
n_presented = (wt_results["presentation_percentile"] <= 2).sum()
print("n presented at 2% threshold:", n_presented, "out of", len(wt_results))
print("percent:", n_presented / len(wt_results))

n presented at 2% threshold: 3778 out of 6520
percent: 0.5794478527607362


In [14]:
#combine and safe
combined_pred = pd.DataFrame({
    "epitope_id": predict_input["epitope_id"].values,
    "hla": predict_input["hla"].values,
    "label": predict_input["label"].values,
    "mt_seq": predict_input["mt_seq"].values,
    "wt_seq": predict_input["wt_seq"].values,
    "mt_length": predict_input["mt_length"].values,

    #mt predictions
    "mt_affinity": mt_results["affinity"].values,
    "mt_affinity_percentile": mt_results["affinity_percentile"].values,
    "mt_presentation_score": mt_results["presentation_score"].values,
    "mt_presentation_percentile": mt_results["presentation_percentile"].values,

    #wt predictions
    "wt_affinity": wt_results["affinity"].values,
    "wt_affinity_percentile": wt_results["affinity_percentile"].values,
    "wt_presentation_score": wt_results["presentation_score"].values,
    "wt_presentation_percentile": wt_results["presentation_percentile"].values,
})

combined_pred.head()

,epitope_id,hla,label,mt_seq,wt_seq,mt_length,mt_affinity,mt_affinity_percentile,mt_presentation_score,mt_presentation_percentile,wt_affinity,wt_affinity_percentile,wt_presentation_score,wt_presentation_percentile
0,551,HLA-A*02:01,1,ACDPHSGHFV,ARDPHSGHFV,10,3125.736683,2.796875,0.053443,4.807092,13379.793366,7.050250,0.029345,8.269212
1,3147,HLA-A*02:01,1,AMLGTHTMEV,AMLSPHAMDV,10,37.507257,0.260250,0.768811,0.326821,104.448134,0.625750,0.483023,0.858478
2,14672,HLA-A*01:01,1,EVDPIGHLY,EVVPISHLY,9,33.501012,0.007125,0.983180,0.004511,164.471650,0.231000,0.941244,0.060136
3,14672,HLA-A*11:01,1,EVDPIGHLY,EVVPISHLY,9,942.095939,1.615250,0.695778,0.443832,1086.646020,1.703250,0.719056,0.406060
4,14672,HLA-B*35:01,1,EVDPIGHLY,EVVPISHLY,9,64.056180,0.173750,0.968890,0.021413,50.299185,0.111375,0.980633,0.007255


In [15]:
#flags for presented/not presented

combined_pred["mt_presented"] = (combined_pred["mt_presentation_percentile"] <= 2).astype(int)
combined_pred["wt_presented"] = (combined_pred["wt_presentation_percentile"] <= 2).astype(int)
combined_pred.head()

,epitope_id,hla,label,mt_seq,wt_seq,mt_length,mt_affinity,mt_affinity_percentile,mt_presentation_score,mt_presentation_percentile,wt_affinity,wt_affinity_percentile,wt_presentation_score,wt_presentation_percentile,mt_presented,wt_presented
0,551,HLA-A*02:01,1,ACDPHSGHFV,ARDPHSGHFV,10,3125.736683,2.796875,0.053443,4.807092,13379.793366,7.050250,0.029345,8.269212,0,0
1,3147,HLA-A*02:01,1,AMLGTHTMEV,AMLSPHAMDV,10,37.507257,0.260250,0.768811,0.326821,104.448134,0.625750,0.483023,0.858478,1,1
2,14672,HLA-A*01:01,1,EVDPIGHLY,EVVPISHLY,9,33.501012,0.007125,0.983180,0.004511,164.471650,0.231000,0.941244,0.060136,1,1
3,14672,HLA-A*11:01,1,EVDPIGHLY,EVVPISHLY,9,942.095939,1.615250,0.695778,0.443832,1086.646020,1.703250,0.719056,0.406060,1,1
4,14672,HLA-B*35:01,1,EVDPIGHLY,EVVPISHLY,9,64.056180,0.173750,0.968890,0.021413,50.299185,0.111375,0.980633,0.007255,1,1


In [16]:
#mutation effect on binding
combined_pred["delta_affinity"] = combined_pred["mt_affinity"] - combined_pred["wt_affinity"]
combined_pred["delta_presentation_score"] = combined_pred["mt_presentation_score"] - combined_pred["wt_presentation_score"]
combined_pred.head()

,epitope_id,hla,label,mt_seq,wt_seq,mt_length,mt_affinity,mt_affinity_percentile,mt_presentation_score,mt_presentation_percentile,wt_affinity,wt_affinity_percentile,wt_presentation_score,wt_presentation_percentile,mt_presented,wt_presented,delta_affinity,delta_presentation_score
0,551,HLA-A*02:01,1,ACDPHSGHFV,ARDPHSGHFV,10,3125.736683,2.796875,0.053443,4.807092,13379.793366,7.050250,0.029345,8.269212,0,0,-10254.056683,0.024097
1,3147,HLA-A*02:01,1,AMLGTHTMEV,AMLSPHAMDV,10,37.507257,0.260250,0.768811,0.326821,104.448134,0.625750,0.483023,0.858478,1,1,-66.940877,0.285789
2,14672,HLA-A*01:01,1,EVDPIGHLY,EVVPISHLY,9,33.501012,0.007125,0.983180,0.004511,164.471650,0.231000,0.941244,0.060136,1,1,-130.970638,0.041936
3,14672,HLA-A*11:01,1,EVDPIGHLY,EVVPISHLY,9,942.095939,1.615250,0.695778,0.443832,1086.646020,1.703250,0.719056,0.406060,1,1,-144.550081,-0.023279
4,14672,HLA-B*35:01,1,EVDPIGHLY,EVVPISHLY,9,64.056180,0.173750,0.968890,0.021413,50.299185,0.111375,0.980633,0.007255,1,1,13.756994,-0.011743


In [17]:
#save file
combined_pred.to_csv("../../processed/cedar_predicted.csv", index=False)